# Indoor Scene Change Detection — Training & Evaluation Pipeline


## 1. Environment Setup


In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q
import os, json, shutil, glob, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print("Libraries ready.")


## 6. Train RT-DETR (comparison model)


In [ ]:
!yolo detect train data=/content/data.yaml model=rtdetr-l.pt epochs={EPOCHS} imgsz={IMGSZ} batch=10 \
    project=/content/runs name=rtdetr


In [ ]:
RTDETR_WEIGHTS = '/content/runs/rtdetr/weights/best.pt'
assert os.path.exists(RTDETR_WEIGHTS), "RT-DETR training did not produce best.pt — check the training log above."
print("RT-DETR weights:", RTDETR_WEIGHTS)


### 6b. RT-DETR Training Curves


In [ ]:
# Section 6b: RT-DETR Training Curves (YOLO Color Style)

rtdetr_results_csv = '/content/runs/rtdetr/results.csv'

if os.path.exists(rtdetr_results_csv):
    rtdetr_hist = pd.read_csv(rtdetr_results_csv)
    rtdetr_hist.columns = [c.strip() for c in rtdetr_hist.columns]

    print("="*60)
    print("RT-DETR Results CSV - Available Columns:")
    print("="*60)
    print(rtdetr_hist.columns.tolist())
    print("="*60)
    print(f"Total epochs: {len(rtdetr_hist)}")
    print()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ----- Plot 1: GIoU Loss (YOLO-র box_loss-এর মতো) -----
    # YOLO Color: Train = Blue, Val = Orange
    axes[0].plot(rtdetr_hist['epoch'], rtdetr_hist['train/giou_loss'],
                 label='train/giou_loss', color='#1f77b4', linewidth=2)  # Blue (YOLO Train)
    axes[0].plot(rtdetr_hist['epoch'], rtdetr_hist['val/giou_loss'],
                 label='val/giou_loss', color='#ff7f0e', linewidth=2, linestyle='--')  # Orange (YOLO Val)

    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('RT-DETR -- Box Loss', fontsize=14)  # YOLO-র মতো Same Title
    axes[0].legend(loc='upper right')
    axes[0].grid(alpha=0.3)

    # ----- Plot 2: mAP (Same as YOLO) -----
    # YOLO Color: mAP50 = Blue, mAP50-95 = Green
    axes[1].plot(rtdetr_hist['epoch'], rtdetr_hist['metrics/mAP50(B)'],
                 label='mAP50', color='#1f77b4', linewidth=2)  # Blue (YOLO mAP50)
    axes[1].plot(rtdetr_hist['epoch'], rtdetr_hist['metrics/mAP50-95(B)'],
                 label='mAP50-95', color='#2ca02c', linewidth=2)  # Green (YOLO mAP50-95)

    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('mAP', fontsize=12)
    axes[1].set_title('RT-DETR -- Validation mAP', fontsize=14)
    axes[1].set_ylim(0, 1.0)
    axes[1].legend(loc='lower right')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/rtdetr_training_curves.png', dpi=150)
    plt.show()

    # ===== Print Summary =====
    print("\n" + "="*60)
    print("RT-DETR Training Summary (Latest Values):")
    print("="*60)
    last_row = rtdetr_hist.iloc[-1]
    print(f"Epoch: {int(last_row['epoch'])}")
    print(f"  train/giou_loss: {last_row['train/giou_loss']:.4f}")
    print(f"  val/giou_loss: {last_row['val/giou_loss']:.4f}")
    print(f"  mAP50: {last_row['metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95: {last_row['metrics/mAP50-95(B)']:.4f}")
    print("="*60)

else:
    print(f"Couldn't find {rtdetr_results_csv}")
    print("Please run Section 6 (RT-DETR training) first.")